In [4]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt
# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')

In [5]:
in_df = client.materialize.query_table('all_neuron_label'
#                                       , filter_equal_dict = {'tag': 'p'}
                                      )

Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


In [6]:
superficial_in = in_df[in_df['pt_position'].apply(lambda coord: coord[1] < 3130)]

In [7]:
superficial_in 

,id,created,superceded_id,valid,tag,pt_supervoxel_id,pt_root_id,pt_position
5,865,2025-06-11 03:35:17.150128+00:00,NaN,t,v,75294763770859416,720575940891783079,"[15966, 3001, 2776]"
8,2083,2025-06-11 03:35:17.864564+00:00,NaN,t,v,77968775982502313,720575940889785957,"[25791, 3004, 2641]"
9,2662,2025-06-25 19:26:30.723067+00:00,NaN,t,p,76913107179542244,720575940899779727,"[21958, 2308, 2261]"
12,2663,2025-06-25 19:26:30.723744+00:00,NaN,t,p,76842738569552584,720575940888561868,"[21576, 2354, 2436]"
13,2715,2025-06-26 16:54:40.514617+00:00,NaN,t,u,78672325515558073,720575940928281596,"[28280, 2523, 1731]"
...,...,...,...,...,...,...,...,...
2585,2428,2025-06-11 03:35:18.067332+00:00,NaN,t,o,73817020143074523,720575940893875701,"[10618, 2908, 2740]"
2625,2476,2025-06-11 03:35:18.095696+00:00,NaN,t,v,77053845137639087,720575940917844107,"[22345, 2455, 3137]"
2633,2486,2025-06-11 03:35:18.101630+00:00,NaN,t,v,76350295470397095,720575940906598846,"[19787, 2907, 3776]"
2645,2368,2025-06-11 03:35:18.031622+00:00,NaN,t,u,77405756638513492,720575940881936635,"[23715, 2711, 1348]"


In [8]:
# All neurons tagged as 'p'
p_neurons = superficial_in[superficial_in["tag"] == "p"].copy()

print(f"Number of 'p' neurons: {len(p_neurons)}")

p_neurons

Number of 'p' neurons: 44


,id,created,superceded_id,valid,tag,pt_supervoxel_id,pt_root_id,pt_position
9,2662,2025-06-25 19:26:30.723067+00:00,NaN,t,p,76913107179542244,720575940899779727,"[21958, 2308, 2261]"
12,2663,2025-06-25 19:26:30.723744+00:00,NaN,t,p,76842738569552584,720575940888561868,"[21576, 2354, 2436]"
25,2679,2025-06-25 23:11:08.446582+00:00,NaN,t,p,73324301629119450,720575940918311989,"[8943, 2467, 2972]"
27,2712,2025-06-25 23:11:08.895338+00:00,NaN,t,p,73535613080578760,720575940927796016,"[9633, 3120, 1219]"
30,2708,2025-06-25 23:11:08.847854+00:00,NaN,t,p,76350225408769910,720575940886290596,"[19801, 2613, 1251]"
31,2667,2025-06-25 23:11:08.267433+00:00,NaN,t,p,81065069043119138,720575940964370854,"[37041, 3108, 1790]"
32,2666,2025-06-25 23:11:08.253838+00:00,NaN,t,p,82050231461686839,720575940893329739,"[40673, 3103, 1892]"
33,2675,2025-06-25 23:11:08.386099+00:00,NaN,t,p,74520708255985251,720575940892250110,"[13304, 2866, 4018]"
34,2668,2025-06-25 23:11:08.281301+00:00,NaN,t,p,78320481794620229,720575940889967848,"[26998, 2358, 1654]"
35,2664,2025-06-25 23:11:08.216439+00:00,NaN,t,p,80220574655328258,720575940884015788,"[33944, 2830, 457]"


In [13]:
import os
import gc
import json
import time
import traceback
import multiprocessing as mp

import numpy as np
import pandas as pd
import trimesh
import pymeshfix

from tqdm.auto import tqdm
from shapely.geometry import Polygon, Point


# ============================================================
# Configuration
# ============================================================

VOXEL_RES_NM = np.array([32.0, 32.0, 45.0])

VOLUME_COLUMN = "soma_volume_15um_um3"
AREA_COLUMN = "soma_area_xy_15um_um2"

STATUS_COLUMN = "soma15_status"

PROGRESS_SUFFIX = ".progress.json"
RETRY_SUFFIX = ".retries.json"


# ============================================================
# Utility functions
# ============================================================

import ast
import json
import re
import numpy as np
import pandas as pd


def parse_position(position):
    """
    Parse a three-coordinate soma position from:
        [123, 456, 789]
        "[123, 456, 789]"
        "[123 456 789]"
        NumPy arrays
        Python lists
    """

    if isinstance(position, str):
        text = position.strip()

        # First try standard JSON.
        try:
            position = json.loads(text)

        except json.JSONDecodeError:
            # Then try a normal Python list/tuple.
            try:
                position = ast.literal_eval(text)

            except (ValueError, SyntaxError):
                # Finally extract three numeric values from strings
                # such as "[123 456 789]".
                values = re.findall(
                    r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?",
                    text,
                )

                if len(values) != 3:
                    raise ValueError(
                        "Could not parse pt_position. "
                        f"Received: {position!r}"
                    )

                position = [
                    float(value)
                    for value in values
                ]

    position = np.asarray(
        position,
        dtype=float,
    ).reshape(-1)

    if position.size != 3:
        raise ValueError(
            "pt_position must contain exactly three coordinates; "
            f"received {position.size}: {position}"
        )

    if not np.all(np.isfinite(position)):
        raise ValueError(
            f"pt_position contains non-finite values: {position}"
        )

    return position


def atomic_save_csv(df, output_file):
    """
    Safely save a CSV without risking a partially written output file.
    """

    temporary_file = output_file + ".tmp"

    df.to_csv(
        temporary_file,
        index=False,
    )

    os.replace(
        temporary_file,
        output_file,
    )


def write_json_atomic(data, filename):
    """
    Safely write a small JSON status file.
    """

    temporary_file = filename + ".tmp"

    with open(temporary_file, "w") as handle:
        json.dump(data, handle)

    os.replace(
        temporary_file,
        filename,
    )


def read_json_safe(filename, default=None):
    """
    Read JSON, returning default if the file is absent or invalid.
    """

    if default is None:
        default = {}

    if not os.path.exists(filename):
        return default

    try:
        with open(filename, "r") as handle:
            return json.load(handle)
    except Exception:
        return default


# ============================================================
# XY area from a selected soma mesh
# ============================================================

def calculate_xy_area_at_soma_z(
    mesh,
    soma_coord_nm,
):
    """
    Intersect the selected soma mesh with the XY plane passing through
    the annotated soma Z coordinate.

    The largest valid closed polygon is used. Because this is applied
    only to the largest soma-containing connected component, the largest
    section polygon should normally correspond to the soma rather than a
    small neurite.

    Returns
    -------
    dict
        area_um2
        area_status
        number_of_polygons
    """

    output = {
        "area_um2": np.nan,
        "area_status": "failed",
        "number_of_polygons": 0,
    }

    section = mesh.section(
        plane_origin=np.asarray(soma_coord_nm, dtype=float),
        plane_normal=np.array([0.0, 0.0, 1.0]),
    )

    if section is None:
        output["area_status"] = "no_xy_intersection"
        return output

    polygons = []

    for path_3d in section.discrete:
        path_3d = np.asarray(path_3d)

        if path_3d.ndim != 2 or len(path_3d) < 4:
            continue

        path_xy = path_3d[:, :2]

        # The section should form a closed path.
        closing_distance_nm = np.linalg.norm(
            path_xy[0] - path_xy[-1]
        )

        if closing_distance_nm > 200:
            continue

        polygon = Polygon(path_xy)

        if not polygon.is_valid:
            polygon = polygon.buffer(0)

        if polygon.is_empty:
            continue

        # buffer(0) can return a MultiPolygon.
        if polygon.geom_type == "MultiPolygon":
            for subpolygon in polygon.geoms:
                if (
                    not subpolygon.is_empty
                    and subpolygon.area > 0
                ):
                    polygons.append(subpolygon)

        elif polygon.geom_type == "Polygon":
            if polygon.area > 0:
                polygons.append(polygon)

    output["number_of_polygons"] = len(polygons)

    if len(polygons) == 0:
        output["area_status"] = "no_closed_xy_polygon"
        return output

    soma_point = Point(
        float(soma_coord_nm[0]),
        float(soma_coord_nm[1]),
    )

    containing_polygons = [
        polygon
        for polygon in polygons
        if polygon.covers(soma_point)
    ]

    if containing_polygons:
        selected_polygon = max(
            containing_polygons,
            key=lambda polygon: polygon.area,
        )

        selection_status = "contains_soma_xy"

    else:
        # The annotated coordinate can be slightly offset from the
        # geometric center. In that case, select the largest section
        # polygon from the already-selected soma component.
        selected_polygon = max(
            polygons,
            key=lambda polygon: polygon.area,
        )

        selection_status = "largest_xy_polygon"

    output["area_um2"] = (
        float(selected_polygon.area) / 1e6
    )

    output["area_status"] = (
        "success_" + selection_status
    )

    return output


# ============================================================
# Two-in-one computation for one neuron
# ============================================================

def compute_soma_volume_and_area_15um(
    root_id,
    coord_voxel,
    radius_um=15,
):
    """
    Compute both soma volume and soma XY area.

    Volume method
    -------------
    1. Load the neuron mesh through mm.mesh().
    2. Apply a 15 µm spherical crop.
    3. Split the crop into connected components.
    4. Select the largest connected component by face count.
    5. Run PyMeshFix on that component when needed.
    6. Use the repaired mesh volume.
    7. Never use convex hull as a fallback.

    Area method
    -----------
    Intersect the selected repaired soma mesh with the XY plane passing
    through the annotated soma Z coordinate and measure the largest valid
    closed cross-sectional polygon.
    """

    root_id = int(root_id)
    radius_nm = float(radius_um) * 1000.0

    coord_voxel = parse_position(coord_voxel)

    soma_coord_nm = (
        coord_voxel * VOXEL_RES_NM
    )

    result = {
        "root_id": root_id,

        "volume_um3": np.nan,
        "area_um2": np.nan,

        "status": "failed",
        "area_status": "not_attempted",

        "watertight_before": np.nan,
        "watertight_after": np.nan,

        "raw_volume_um3": np.nan,
        "repair_volume_um3": np.nan,

        "bbox_x_um": np.nan,
        "bbox_y_um": np.nan,
        "bbox_z_um": np.nan,

        "component_faces": np.nan,
        "component_vertices": np.nan,
        "n_components": np.nan,

        "area_polygon_count": np.nan,
        "error": None,
    }

    seg_mesh = None
    vertices = None
    faces = None
    crop_mesh = None
    components = None
    soma_mesh = None
    repaired_mesh = None
    meshfix = None

    try:
        # Uses cached H5 mesh when available.
        seg_mesh = mm.mesh(
            seg_id=root_id
        )

        vertices = np.asarray(
            seg_mesh.vertices
        )

        faces = np.asarray(
            seg_mesh.faces
        )

        # ----------------------------------------------------
        # Spherical crop
        # ----------------------------------------------------

        displacement = vertices - soma_coord_nm

        distance_squared = np.einsum(
            "ij,ij->i",
            displacement,
            displacement,
        )

        vertex_mask = (
            distance_squared < radius_nm ** 2
        )

        face_mask = (
            vertex_mask[faces].all(axis=1)
        )

        selected_faces = faces[face_mask]

        if len(selected_faces) == 0:
            result["status"] = "no_faces_in_crop"
            return result

        selected_vertex_indices, inverse = np.unique(
            selected_faces.ravel(),
            return_inverse=True,
        )

        crop_mesh = trimesh.Trimesh(
            vertices=vertices[selected_vertex_indices],
            faces=inverse.reshape((-1, 3)),
            process=False,
        )

        crop_mesh.remove_unreferenced_vertices()

        # ----------------------------------------------------
        # Largest connected component
        # ----------------------------------------------------

        components = crop_mesh.split(
            only_watertight=False,
            engine=None,
        )

        result["n_components"] = len(components)

        if len(components) == 0:
            result["status"] = "no_components"
            return result

        soma_mesh = max(
            components,
            key=lambda component: len(component.faces),
        )

        bbox_um = (
            np.asarray(
                soma_mesh.bounding_box.extents,
                dtype=float,
            )
            / 1000.0
        )

        result["bbox_x_um"] = float(bbox_um[0])
        result["bbox_y_um"] = float(bbox_um[1])
        result["bbox_z_um"] = float(bbox_um[2])

        result["component_faces"] = int(
            len(soma_mesh.faces)
        )

        result["component_vertices"] = int(
            len(soma_mesh.vertices)
        )

        result["watertight_before"] = bool(
            soma_mesh.is_watertight
        )

        result["raw_volume_um3"] = (
            abs(float(soma_mesh.volume)) / 1e9
        )

        # ----------------------------------------------------
        # Repair the selected largest component
        # ----------------------------------------------------

        if soma_mesh.is_watertight:
            repaired_mesh = soma_mesh

        else:
            meshfix = pymeshfix.MeshFix(
                np.asarray(soma_mesh.vertices),
                np.asarray(soma_mesh.faces),
            )

            meshfix.repair(
                joincomp=True,
                remove_smallest_components=False,
                verbose=False,
            )

            repaired_mesh = trimesh.Trimesh(
                vertices=meshfix.v,
                faces=meshfix.f,
                process=False,
            )

        result["watertight_after"] = bool(
            repaired_mesh.is_watertight
        )

        repaired_volume_um3 = (
            abs(float(repaired_mesh.volume)) / 1e9
        )

        result["repair_volume_um3"] = (
            repaired_volume_um3
        )

        if not np.isfinite(repaired_volume_um3):
            result["status"] = (
                "nonfinite_repaired_volume"
            )
            return result

        if repaired_volume_um3 <= 0:
            result["status"] = (
                "nonpositive_repaired_volume"
            )
            return result

        # Use repaired volume even if PyMeshFix does not report a
        # completely watertight result. Keep the flag for QC.
        result["volume_um3"] = repaired_volume_um3

        if repaired_mesh.is_watertight:
            result["status"] = "success_watertight"
        else:
            result["status"] = (
                "success_nonwatertight_qc"
            )

        # ----------------------------------------------------
        # XY cross-sectional area at soma Z
        # ----------------------------------------------------

        area_result = calculate_xy_area_at_soma_z(
            mesh=repaired_mesh,
            soma_coord_nm=soma_coord_nm,
        )

        result["area_um2"] = area_result[
            "area_um2"
        ]

        result["area_status"] = area_result[
            "area_status"
        ]

        result["area_polygon_count"] = (
            area_result["number_of_polygons"]
        )

        return result

    except Exception as exc:
        result["status"] = (
            "failed_"
            + type(exc).__name__
        )

        result["error"] = (
            type(exc).__name__
            + ": "
            + str(exc)
        )

        return result

    finally:
        meshfix = None
        repaired_mesh = None
        soma_mesh = None
        components = None
        crop_mesh = None
        faces = None
        vertices = None
        seg_mesh = None

        gc.collect()


def update_soma_volume_and_area_15um(
    csv_file="superficial_in_soma_volume.csv",
    output_file="superficial_in_soma_volume_15um.csv",
    radius_um=15,
    skip_root_ids=None,
):
    """
    Safely calculate and update 15-um soma volume and XY area.

    Behavior
    --------
    1. Resume from output_file when it already exists.
    2. Otherwise load csv_file.
    3. Skip root IDs listed in skip_root_ids before loading their meshes.
    4. Preserve completed rows.
    5. Save atomically after every row.
    6. Record errors without stopping the full batch.
    """

    # --------------------------------------------------------
    # Normalize skip IDs
    # --------------------------------------------------------
    if skip_root_ids is None:
        skip_root_ids = set()

    skip_root_ids = {
        int(root_id)
        for root_id in skip_root_ids
    }

    # --------------------------------------------------------
    # Safely choose and load input
    # --------------------------------------------------------
    if os.path.exists(output_file):
        input_file = output_file
        print(f"Resuming from existing output: {output_file}")

    elif os.path.exists(csv_file):
        input_file = csv_file
        print(f"Starting from source CSV: {csv_file}")

    else:
        raise FileNotFoundError(
            "Neither the source CSV nor the output CSV exists.\n"
            f"Source: {csv_file}\n"
            f"Output: {output_file}"
        )

    try:
        df = pd.read_csv(
            input_file,
            dtype={"root_id": "string"},
            low_memory=False,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Could not safely read CSV file: {input_file}"
        ) from exc

    # --------------------------------------------------------
    # Validate required columns
    # --------------------------------------------------------
    required_columns = {
        "root_id",
        "pt_position",
    }

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            "CSV is missing required columns: "
            + ", ".join(sorted(missing_columns))
        )

    # Keep root IDs as nullable unsigned integers.
    df["root_id"] = pd.to_numeric(
        df["root_id"],
        errors="coerce",
    ).astype("UInt64")

    # --------------------------------------------------------
    # Initialize output columns
    # --------------------------------------------------------
    numeric_columns = [
        VOLUME_COLUMN,
        AREA_COLUMN,
        "soma15_raw_volume_um3",
        "soma15_repair_volume_um3",
        "soma15_bbox_x_um",
        "soma15_bbox_y_um",
        "soma15_bbox_z_um",
        "soma15_component_faces",
        "soma15_component_vertices",
        "soma15_n_components",
        "soma15_area_polygon_count",
    ]

    for column in numeric_columns:
        if column not in df.columns:
            df[column] = np.nan
        else:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

    text_columns = [
        STATUS_COLUMN,
        "soma15_area_status",
        "soma15_error",
    ]

    for column in text_columns:
        if column not in df.columns:
            df[column] = ""
        else:
            df[column] = (
                df[column]
                .fillna("")
                .astype(str)
            )

    bool_columns = [
        "soma15_watertight_before",
        "soma15_watertight_after",
    ]

    for column in bool_columns:
        if column not in df.columns:
            df[column] = np.nan

    # --------------------------------------------------------
    # Mark all requested skip IDs immediately
    # --------------------------------------------------------
    valid_root_mask = df["root_id"].notna()

    skip_mask = pd.Series(
        False,
        index=df.index,
    )

    if skip_root_ids:
        skip_mask = (
            valid_root_mask
            & df["root_id"].isin(skip_root_ids)
        )

        for idx in df.index[skip_mask]:
            root_id = int(df.at[idx, "root_id"])

            df.at[idx, STATUS_COLUMN] = (
                "skipped_problematic_neuron"
            )

            df.at[idx, "soma15_area_status"] = (
                "skipped_problematic_neuron"
            )

            df.at[idx, "soma15_error"] = (
                f"Root ID {root_id} was intentionally skipped."
            )

        if skip_mask.any():
            print(
                "Marked skipped neurons:",
                [
                    int(value)
                    for value in df.loc[
                        skip_mask,
                        "root_id",
                    ].tolist()
                ],
            )

            atomic_save_csv(
                df,
                output_file,
            )

    # --------------------------------------------------------
    # Determine which rows still require processing
    # --------------------------------------------------------
    volume_complete = np.isfinite(
        pd.to_numeric(
            df[VOLUME_COLUMN],
            errors="coerce",
        )
    )

    area_complete = np.isfinite(
        pd.to_numeric(
            df[AREA_COLUMN],
            errors="coerce",
        )
    )

    status_text = (
        df[STATUS_COLUMN]
        .fillna("")
        .astype(str)
    )

    terminal_status = (
        status_text.str.startswith("skipped_")
        | status_text.str.startswith("failed_timeout")
    )

    todo_mask = (
        ~(volume_complete & area_complete)
        & ~terminal_status
    )

    todo_indices = df.index[todo_mask].tolist()

    print(f"Total rows: {len(df)}")
    print(f"Rows remaining: {len(todo_indices)}")

    # --------------------------------------------------------
    # Process rows individually
    # --------------------------------------------------------
    for idx in tqdm(
        todo_indices,
        desc="Calculating soma volume and area",
    ):
        root_value = df.at[idx, "root_id"]

        # ----------------------------------------------------
        # Invalid or missing root ID
        # ----------------------------------------------------
        if pd.isna(root_value):
            df.at[idx, STATUS_COLUMN] = (
                "failed_invalid_root_id"
            )

            df.at[idx, "soma15_area_status"] = (
                "not_attempted"
            )

            df.at[idx, "soma15_error"] = (
                "root_id is missing or could not be converted "
                "to an integer."
            )

            atomic_save_csv(
                df,
                output_file,
            )

            continue

        root_id = int(root_value)

        # Extra guard in case the dataframe changed after todo creation.
        if root_id in skip_root_ids:
            df.at[idx, STATUS_COLUMN] = (
                "skipped_problematic_neuron"
            )

            df.at[idx, "soma15_area_status"] = (
                "skipped_problematic_neuron"
            )

            df.at[idx, "soma15_error"] = (
                f"Root ID {root_id} was intentionally skipped."
            )

            atomic_save_csv(
                df,
                output_file,
            )

            continue

        print(f"\nProcessing root ID: {root_id}")

        try:
            # ------------------------------------------------
            # Validate soma coordinate
            # ------------------------------------------------
            position_value = df.at[idx, "pt_position"]

            if pd.isna(position_value):
                raise ValueError(
                    "pt_position is missing."
                )

            coord = parse_position(
                position_value
            )

            if not np.all(np.isfinite(coord)):
                raise ValueError(
                    "pt_position contains non-finite values."
                )

            # ------------------------------------------------
            # Run mesh calculation
            # ------------------------------------------------
            result = compute_soma_volume_and_area_15um(
                root_id=root_id,
                coord_voxel=coord,
                radius_um=radius_um,
            )

            # ------------------------------------------------
            # Store result
            # ------------------------------------------------
            df.at[idx, VOLUME_COLUMN] = (
                result.get("volume_um3", np.nan)
            )

            df.at[idx, AREA_COLUMN] = (
                result.get("area_um2", np.nan)
            )

            df.at[idx, STATUS_COLUMN] = (
                result.get("status", "failed_unknown")
            )

            df.at[idx, "soma15_area_status"] = (
                result.get(
                    "area_status",
                    "not_attempted",
                )
            )

            df.at[idx, "soma15_raw_volume_um3"] = (
                result.get(
                    "raw_volume_um3",
                    np.nan,
                )
            )

            df.at[idx, "soma15_repair_volume_um3"] = (
                result.get(
                    "repair_volume_um3",
                    np.nan,
                )
            )

            df.at[idx, "soma15_watertight_before"] = (
                result.get(
                    "watertight_before",
                    np.nan,
                )
            )

            df.at[idx, "soma15_watertight_after"] = (
                result.get(
                    "watertight_after",
                    np.nan,
                )
            )

            df.at[idx, "soma15_bbox_x_um"] = (
                result.get("bbox_x_um", np.nan)
            )

            df.at[idx, "soma15_bbox_y_um"] = (
                result.get("bbox_y_um", np.nan)
            )

            df.at[idx, "soma15_bbox_z_um"] = (
                result.get("bbox_z_um", np.nan)
            )

            df.at[idx, "soma15_component_faces"] = (
                result.get(
                    "component_faces",
                    np.nan,
                )
            )

            df.at[idx, "soma15_component_vertices"] = (
                result.get(
                    "component_vertices",
                    np.nan,
                )
            )

            df.at[idx, "soma15_n_components"] = (
                result.get(
                    "n_components",
                    np.nan,
                )
            )

            df.at[idx, "soma15_area_polygon_count"] = (
                result.get(
                    "area_polygon_count",
                    np.nan,
                )
            )

            error_value = result.get("error")

            df.at[idx, "soma15_error"] = (
                ""
                if error_value is None
                else str(error_value)
            )

        except Exception as exc:
            error_message = (
                f"{type(exc).__name__}: {exc}"
            )

            print(
                f"Failed root ID {root_id}: "
                f"{error_message}"
            )

            df.at[idx, STATUS_COLUMN] = (
                "failed_"
                + type(exc).__name__
            )

            df.at[idx, "soma15_area_status"] = (
                "not_attempted"
            )

            df.at[idx, "soma15_error"] = (
                error_message
            )

        finally:
            # Save after every row, whether successful or failed.
            atomic_save_csv(
                df,
                output_file,
            )

            gc.collect()

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------
    final_status = (
        df[STATUS_COLUMN]
        .fillna("")
        .astype(str)
    )

    print("\nProcessing finished.")
    print(
        "Successful rows:",
        int(final_status.str.startswith("success_").sum()),
    )
    print(
        "Skipped rows:",
        int(final_status.str.startswith("skipped_").sum()),
    )
    print(
        "Failed rows:",
        int(final_status.str.startswith("failed_").sum()),
    )
    print("Output saved to:", output_file)

    return df
# ============================================================
# Completion checker
# ============================================================

def soma_processing_complete(
    output_file,
):
    """
    Return True when every row has both a valid volume and area,
    or has reached a terminal timeout status.
    """

    if not os.path.exists(output_file):
        return False

    df = pd.read_csv(
        output_file,
        dtype={"root_id": "string"},
    )

    if VOLUME_COLUMN not in df.columns:
        return False

    if AREA_COLUMN not in df.columns:
        return False

    volume_complete = np.isfinite(
        pd.to_numeric(
            df[VOLUME_COLUMN],
            errors="coerce",
        )
    )

    area_complete = np.isfinite(
        pd.to_numeric(
            df[AREA_COLUMN],
            errors="coerce",
        )
    )

    if STATUS_COLUMN in df.columns:
        terminal_timeout = (
            df[STATUS_COLUMN]
            .fillna("")
            .astype(str)
            .str.startswith("failed_timeout")
        )
    else:
        terminal_timeout = pd.Series(
            False,
            index=df.index,
        )

    return bool(
        (
            (volume_complete & area_complete)
            | terminal_timeout
        ).all()
    )


# ============================================================
# Watchdog support
# ============================================================

def mark_timed_out_neuron(
    output_file,
    max_retries=3,
):
    """
    Record a timeout for the neuron named in the progress JSON.

    After max_retries, the neuron is marked as a terminal failure so the
    overnight process can continue with the remaining neurons.
    """

    progress_file = (
        output_file + PROGRESS_SUFFIX
    )

    retry_file = (
        output_file + RETRY_SUFFIX
    )

    progress = read_json_safe(
        progress_file,
        default={},
    )

    if "root_id" not in progress:
        return

    root_id = int(progress["root_id"])

    retries = read_json_safe(
        retry_file,
        default={},
    )

    retry_key = str(root_id)

    retry_count = int(
        retries.get(retry_key, 0)
    ) + 1

    retries[retry_key] = retry_count

    write_json_atomic(
        retries,
        retry_file,
    )

    if not os.path.exists(output_file):
        return

    df = pd.read_csv(
        output_file,
        dtype={"root_id": "string"},
    )

    root_ids = pd.to_numeric(
        df["root_id"],
        errors="coerce",
    )

    matching_indices = df.index[
        root_ids == root_id
    ].tolist()

    if not matching_indices:
        return

    idx = matching_indices[0]

    if STATUS_COLUMN not in df.columns:
        df[STATUS_COLUMN] = pd.Series(
            index=df.index,
            dtype="object",
        )

    if retry_count >= max_retries:
        df.at[
            idx,
            STATUS_COLUMN,
        ] = (
            f"failed_timeout_after_"
            f"{retry_count}_attempts"
        )
    else:
        df.at[
            idx,
            STATUS_COLUMN,
        ] = (
            f"timeout_retry_"
            f"{retry_count}_of_{max_retries}"
        )

    atomic_save_csv(
        df,
        output_file,
    )


def _batch_worker_entry(
    csv_file,
    output_file,
    radius_um,
):
    """
    Entry point used by the watchdog process.
    """

    update_soma_volume_and_area_15um(
        csv_file=csv_file,
        output_file=output_file,
        radius_um=radius_um,
    )


# ============================================================
# Overnight supervisor
# ============================================================

def run_soma_volume_area_overnight(
    csv_file="superficial_in_soma_volume.csv",
    output_file="superficial_in_soma_volume_15um.csv",
    radius_um=15,
    stale_minutes=10,
    poll_seconds=30,
    max_timeout_retries=3,
):
    """
    Run the analysis overnight with automatic restart.

    The worker saves the output after every neuron. If the output CSV has
    not changed for `stale_minutes`, the worker is considered stuck. It
    is terminated and restarted from the existing output CSV.

    On macOS/Jupyter this uses the 'fork' multiprocessing context so the
    existing `mm` MeshMeta object is inherited by the worker.
    """

    stale_seconds = (
        float(stale_minutes) * 60.0
    )

    context = mp.get_context("fork")

    print("Starting overnight soma analysis.")
    print("Source:", csv_file)
    print("Output:", output_file)
    print(
        "Stale timeout:",
        stale_minutes,
        "minutes",
    )

    while True:
        if soma_processing_complete(
            output_file
        ):
            print(
                "\nAll neurons are complete, "
                "or flagged after repeated timeouts."
            )
            break

        worker = context.Process(
            target=_batch_worker_entry,
            args=(
                csv_file,
                output_file,
                radius_um,
            ),
        )

        worker.start()

        print(
            "\nWorker started with PID:",
            worker.pid,
        )

        if os.path.exists(output_file):
            last_update_time = os.path.getmtime(
                output_file
            )
        else:
            last_update_time = time.time()

        while worker.is_alive():
            time.sleep(poll_seconds)

            if os.path.exists(output_file):
                current_update_time = os.path.getmtime(
                    output_file
                )

                if current_update_time > last_update_time:
                    last_update_time = (
                        current_update_time
                    )

            seconds_since_update = (
                time.time() - last_update_time
            )

            if seconds_since_update >= stale_seconds:
                print(
                    "\nNo CSV update for "
                    f"{stale_minutes} minutes."
                )

                print(
                    "Terminating stuck worker..."
                )

                worker.terminate()
                worker.join(timeout=30)

                if worker.is_alive():
                    worker.kill()
                    worker.join(timeout=10)

                mark_timed_out_neuron(
                    output_file=output_file,
                    max_retries=max_timeout_retries,
                )

                print(
                    "Restarting from saved progress..."
                )

                break

        if not worker.is_alive():
            worker.join()

            if soma_processing_complete(
                output_file
            ):
                print(
                    "\nProcessing completed."
                )
                break

            print(
                "\nWorker exited before all rows "
                "were complete. Restarting..."
            )

            time.sleep(5)

    final_df = pd.read_csv(
        output_file,
        dtype={"root_id": "string"},
    )

    return final_df

In [9]:
import os
import time
import numpy as np
import pandas as pd
import trimesh
import pymeshfix

from meshparty import trimesh_io
from tqdm.auto import tqdm


# ============================================================
# Configuration
# ============================================================

SEG_VOL_PATH = (
    "graphene://https://cave.fanc-fly.com/segmentation/"
    "table/wclee_mouse_spinalcord_cltmr"
)

CACHE_DIR = "/Users/wangchu/connectivity_analysis/meshes"



# ============================================================
# MeshMeta
# ============================================================

mm = trimesh_io.MeshMeta(
    cv_path=SEG_VOL_PATH,
    disk_cache_path=CACHE_DIR,
    map_gs_to_https=True,
)

In [10]:
# Root IDs that should never be loaded or processed.
SKIP_ROOT_IDS = {
    720575940902438328,
}

results = update_soma_volume_and_area_15um(
    csv_file="superficial_in_soma_volume.csv",
    output_file="superficial_in_soma_volume_15um.csv",
    radius_um=15,
    skip_root_ids=SKIP_ROOT_IDS,
)

# Optional: inspect the status summary.
display(
    results[STATUS_COLUMN]
    .fillna("missing")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="number_of_neurons")
)

Resuming from existing output: superficial_in_soma_volume_15um.csv
Marked skipped neurons: [720575940902438328]
Total rows: 495
Rows remaining: 143


Calculating soma volume and area:   0%|          | 0/143 [00:00<?, ?it/s]


Processing root ID: 720575940892250110

Processing root ID: 720575940896286391

Processing root ID: 720575940885305958



Processing root ID: 720575940868750061



Processing root ID: 720575940882637918


...


Processing root ID: 720575940906598846



Processing root ID: 720575940881936635



Processing root ID: 720575940894216347



Processing finished.
Successful rows: 488
Skipped rows: 1
Failed rows: 6
Output saved to: superficial_in_soma_volume_15um.csv


,status,number_of_neurons
0,success_watertight,433
1,success_nonwatertight_qc,55
2,failed_ProxyError,6
3,skipped_problematic_neuron,1


In [14]:
import pandas as pd
import numpy as np


soma_csv = "superficial_in_soma_volume_15um.csv"
missing_pn_csv = "missing_pn_soma_calculation_input.csv"
output_csv = "superficial_in_soma_volume_15um_updated.csv"


# ------------------------------------------------------------
# Load files
# ------------------------------------------------------------

soma_df = pd.read_csv(
    soma_csv,
    dtype={"root_id": "string"},
    low_memory=False,
)

missing_df = pd.read_csv(
    missing_pn_csv,
    dtype={"root_id": "string"},
)


print("Existing soma CSV:", len(soma_df))
print("New neurons to add:", len(missing_df))


# ------------------------------------------------------------
# Make sure these neurons are not already present
# ------------------------------------------------------------

existing_ids = set(
    pd.to_numeric(
        soma_df["root_id"],
        errors="coerce",
    )
    .dropna()
    .astype("int64")
)

missing_df["root_id_numeric"] = pd.to_numeric(
    missing_df["root_id"],
    errors="raise",
).astype("int64")

missing_df = missing_df.loc[
    ~missing_df["root_id_numeric"].isin(existing_ids)
].copy()

print("Actually missing:", len(missing_df))


# ------------------------------------------------------------
# Compute only the missing neurons
# ------------------------------------------------------------

new_rows = []

for _, row in missing_df.iterrows():

    root_id = int(row["root_id_numeric"])

    coord = parse_position(
        row["pt_position"]
    )

    print(
        f"\nCalculating: {root_id}, "
        f"coordinate: {coord.tolist()}"
    )

    result = compute_soma_volume_and_area_15um(
        root_id=root_id,
        coord_voxel=coord,
        radius_um=15,
    )

    print(
        "Status:",
        result["status"],
        "| volume:",
        result["volume_um3"],
        "| area:",
        result["area_um2"],
    )

    new_row = {
        column: np.nan
        for column in soma_df.columns
    }

    new_row["root_id"] = str(root_id)
    new_row["pt_position"] = json.dumps(
        coord.astype(int).tolist()
    )
    new_row["tag"] = "p"

    new_row[VOLUME_COLUMN] = result["volume_um3"]
    new_row[AREA_COLUMN] = result["area_um2"]

    new_row[STATUS_COLUMN] = result["status"]
    new_row["soma15_area_status"] = result["area_status"]
    new_row["soma15_error"] = result["error"]

    new_row["soma15_raw_volume_um3"] = result["raw_volume_um3"]
    new_row["soma15_repair_volume_um3"] = result["repair_volume_um3"]

    new_row["soma15_watertight_before"] = result["watertight_before"]
    new_row["soma15_watertight_after"] = result["watertight_after"]

    new_row["soma15_bbox_x_um"] = result["bbox_x_um"]
    new_row["soma15_bbox_y_um"] = result["bbox_y_um"]
    new_row["soma15_bbox_z_um"] = result["bbox_z_um"]

    new_row["soma15_component_faces"] = result["component_faces"]
    new_row["soma15_component_vertices"] = result["component_vertices"]
    new_row["soma15_n_components"] = result["n_components"]
    new_row["soma15_area_polygon_count"] = result["area_polygon_count"]

    new_rows.append(new_row)


# ------------------------------------------------------------
# Only append successful calculations
# ------------------------------------------------------------

new_rows_df = pd.DataFrame(new_rows)

successful_new_rows = new_rows_df.loc[
    new_rows_df[STATUS_COLUMN]
    .fillna("")
    .astype(str)
    .str.startswith("success_")
].copy()

failed_new_rows = new_rows_df.loc[
    ~new_rows_df.index.isin(successful_new_rows.index)
].copy()

print("\nSuccessful new rows:", len(successful_new_rows))
print("Failed new rows:", len(failed_new_rows))

if len(failed_new_rows) > 0:
    display(
        failed_new_rows[
            [
                "root_id",
                STATUS_COLUMN,
                "soma15_error",
            ]
        ]
    )


# ------------------------------------------------------------
# Save to a new file; do not overwrite the original
# ------------------------------------------------------------

updated_df = pd.concat(
    [
        soma_df,
        successful_new_rows,
    ],
    ignore_index=True,
)

updated_df.to_csv(
    output_csv,
    index=False,
)

print("\nOriginal rows:", len(soma_df))
print("Appended rows:", len(successful_new_rows))
print("Final rows:", len(updated_df))
print("Saved safely to:", output_csv)

Existing soma CSV: 495
New neurons to add: 3
Actually missing: 3

Calculating: 720575940875863357, coordinate: [43651.0, 3371.0, 1203.0]


Status: success_watertight | volume: 1658.902856063125 | area: 152.96119816223467

Calculating: 720575940902401208, coordinate: [43341.0, 3453.0, 2176.0]


Status: failed_ProxyError | volume: nan | area: nan

Calculating: 720575940930825549, coordinate: [46316.0, 3362.0, 1139.0]
Status: success_watertight | volume: 1773.842866810125 | area: 175.57399037873049

Successful new rows: 2
Failed new rows: 1


,root_id,soma15_status,soma15_error
1,720575940902401208,failed_ProxyError,ProxyError: HTTPSConnectionPool(host='cave.fan...



Original rows: 495
Appended rows: 2
Final rows: 497
Saved safely to: superficial_in_soma_volume_15um_updated.csv


In [22]:
import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# Files
# ------------------------------------------------------------

input_csv = "superficial_in_soma_volume_15um_updated.csv"
output_csv = "superficial_in_soma_volume_15um_updated.csv"

# Remaining neuron
root_id = 720575940920533392
coord = [42266, 3057, 573]


# ------------------------------------------------------------
# Load current updated table
# ------------------------------------------------------------

soma_df = pd.read_csv(
    input_csv,
    dtype={"root_id": "string"},
    low_memory=False,
)

existing_ids = set(
    pd.to_numeric(
        soma_df["root_id"],
        errors="coerce",
    )
    .dropna()
    .astype("int64")
)

if root_id in existing_ids:
    print(f"{root_id} is already present. Nothing to do.")

else:
    print(f"Retrying {root_id}...")

    result = compute_soma_volume_and_area_15um(
        root_id=root_id,
        coord_voxel=coord,
        radius_um=15,
    )

    print("Status:", result["status"])
    print("Volume:", result["volume_um3"])
    print("Area:", result["area_um2"])
    print("Error:", result["error"])

    if str(result["status"]).startswith("success_"):

        new_row = {
            column: np.nan
            for column in soma_df.columns
        }

        new_row["root_id"] = str(root_id)
        new_row["pt_position"] = json.dumps(coord)
        new_row["tag"] = "p"

        new_row[VOLUME_COLUMN] = result["volume_um3"]
        new_row[AREA_COLUMN] = result["area_um2"]

        new_row[STATUS_COLUMN] = result["status"]
        new_row["soma15_area_status"] = result["area_status"]
        new_row["soma15_error"] = result["error"]

        new_row["soma15_raw_volume_um3"] = result["raw_volume_um3"]
        new_row["soma15_repair_volume_um3"] = result["repair_volume_um3"]

        new_row["soma15_watertight_before"] = result["watertight_before"]
        new_row["soma15_watertight_after"] = result["watertight_after"]

        new_row["soma15_bbox_x_um"] = result["bbox_x_um"]
        new_row["soma15_bbox_y_um"] = result["bbox_y_um"]
        new_row["soma15_bbox_z_um"] = result["bbox_z_um"]

        new_row["soma15_component_faces"] = result["component_faces"]
        new_row["soma15_component_vertices"] = result["component_vertices"]
        new_row["soma15_n_components"] = result["n_components"]
        new_row["soma15_area_polygon_count"] = result["area_polygon_count"]

        soma_df = pd.concat(
            [
                soma_df,
                pd.DataFrame([new_row]),
            ],
            ignore_index=True,
        )

        atomic_save_csv(
            soma_df,
            output_csv,
        )

        print(f"\nSuccessfully appended {root_id}")
        print("Final rows:", len(soma_df))
        print("Saved to:", output_csv)

    else:
        print(
            "\nRetry failed again. The CSV was not modified."
        )

Retrying 720575940920533392...


Status: success_watertight
Volume: 1996.4089110410414
Area: 132.1125019375695
Error: None

Successfully appended 720575940920533392
Final rows: 500
Saved to: superficial_in_soma_volume_15um_updated.csv


In [24]:
import pandas as pd

# --------------------------------------------------
# Corrected soma coordinates
# --------------------------------------------------

corrected_soma_positions = {
    720575940888418764: [17132, 2634, 1220],
    720575940894614067: [23113, 2706, 1231],
    720575940898813177: [42496, 3086, 554],
    720575940910981186: [12471, 2685, 906],
}

# --------------------------------------------------
# Recalculate only; do not modify any CSV
# --------------------------------------------------

results_list = []

for root_id, coord in corrected_soma_positions.items():

    print("\n" + "=" * 70)
    print(f"Root ID: {root_id}")
    print(f"New soma coordinate: {coord}")

    result = compute_soma_volume_and_area_15um(
        root_id=root_id,
        coord_voxel=coord,
        radius_um=15,
    )

    print(f"Status: {result['status']}")
    print(f"Volume: {result['volume_um3']:.3f} µm³")
    print(f"XY area: {result['area_um2']:.3f} µm²")
    print(f"Area status: {result['area_status']}")
    print(f"Watertight before: {result['watertight_before']}")
    print(f"Watertight after: {result['watertight_after']}")
    print(f"Error: {result['error']}")

    results_list.append({
        "root_id": root_id,
        "new_pt_position": coord,
        "soma_volume_15um_um3": result["volume_um3"],
        "soma_area_xy_15um_um2": result["area_um2"],
        "soma15_status": result["status"],
        "soma15_area_status": result["area_status"],
        "soma15_watertight_before": result["watertight_before"],
        "soma15_watertight_after": result["watertight_after"],
        "soma15_error": result["error"],
    })

# --------------------------------------------------
# Summary table
# --------------------------------------------------

recalculated_df = pd.DataFrame(results_list)

print("\n\nSummary")
display(
    recalculated_df[
        [
            "root_id",
            "new_pt_position",
            "soma_volume_15um_um3",
            "soma_area_xy_15um_um2",
            "soma15_status",
        ]
    ]
)


Root ID: 720575940888418764
New soma coordinate: [17132, 2634, 1220]
Status: success_watertight
Volume: 1224.711 µm³
XY area: 131.443 µm²
Area status: success_contains_soma_xy
Watertight before: False
Watertight after: True
Error: None

Root ID: 720575940894614067
New soma coordinate: [23113, 2706, 1231]
Status: success_watertight
Volume: 1176.599 µm³
XY area: 106.052 µm²
Area status: success_contains_soma_xy
Watertight before: False
Watertight after: True
Error: None

Root ID: 720575940898813177
New soma coordinate: [42496, 3086, 554]
Status: success_watertight
Volume: 2068.129 µm³
XY area: 141.311 µm²
Area status: success_contains_soma_xy
Watertight before: False
Watertight after: True
Error: None

Root ID: 720575940910981186
New soma coordinate: [12471, 2685, 906]
Status: success_watertight
Volume: 1485.622 µm³
XY area: 112.590 µm²
Area status: success_contains_soma_xy
Watertight before: False
Watertight after: True
Error: None


Summary


,root_id,new_pt_position,soma_volume_15um_um3,soma_area_xy_15um_um2,soma15_status
0,720575940888418764,"[17132, 2634, 1220]",1224.711166,131.443279,success_watertight
1,720575940894614067,"[23113, 2706, 1231]",1176.599136,106.052441,success_watertight
2,720575940898813177,"[42496, 3086, 554]",2068.129325,141.311211,success_watertight
3,720575940910981186,"[12471, 2685, 906]",1485.621975,112.589755,success_watertight


In [27]:
import pandas as pd
import json

CSV_FILE = "superficial_in_soma_volume_15um_updated.csv"

root_id = 720575940894614067

new_position = [23113, 2706, 1231]
new_volume = 1176.599136
new_area = 106.052441


df = pd.read_csv(
    CSV_FILE,
    dtype={"root_id": "string"},
    low_memory=False,
)


# exact string matching (your CSV is already clean)
mask = (
    df["root_id"].astype(str)
    == str(root_id)
)

print("Found rows:", mask.sum())

if mask.sum() != 1:
    raise ValueError(
        f"Expected 1 row, found {mask.sum()}"
    )


# show old values first
print("\nBefore:")
display(
    df.loc[
        mask,
        [
            "root_id",
            "pt_position",
            "soma_volume_15um_um3",
            "soma_area_xy_15um_um2",
        ]
    ]
)


# update
df.loc[mask, "pt_position"] = json.dumps(
    new_position
)

df.loc[mask, "soma_volume_15um_um3"] = (
    new_volume
)

df.loc[mask, "soma_area_xy_15um_um2"] = (
    new_area
)

df.loc[mask, "soma15_status"] = (
    "success_watertight_corrected_coordinate"
)

df.loc[mask, "soma15_area_status"] = (
    "success_corrected_coordinate"
)


# save new copy
output_file = (
    "superficial_in_soma_volume_15um_updated_corrected.csv"
)

df.to_csv(
    output_file,
    index=False,
)


print("\nAfter:")
display(
    df.loc[
        mask,
        [
            "root_id",
            "pt_position",
            "soma_volume_15um_um3",
            "soma_area_xy_15um_um2",
            "soma15_status",
        ]
    ]
)

print("\nSaved:", output_file)

Found rows: 1

Before:


,root_id,pt_position,soma_volume_15um_um3,soma_area_xy_15um_um2
382,720575940894614067,"[23207, 2722, 1142]",1202.545242,121.926715



After:


,root_id,pt_position,soma_volume_15um_um3,soma_area_xy_15um_um2,soma15_status
382,720575940894614067,"[23113, 2706, 1231]",1176.599136,106.052441,success_watertight_corrected_coordinate



Saved: superficial_in_soma_volume_15um_updated_corrected.csv


In [26]:
df = pd.read_csv(
    "superficial_in_soma_volume_15um_updated.csv",
    dtype={"root_id": "string"},
    low_memory=False,
)

print(df["root_id"].head())
print(df["root_id"].dtype)

# search approximately
print(
    df[
        df["root_id"].str.contains(
            "720575940894614067",
            na=False,
        )
    ]
)

0    720575940891783079
1    720575940889785957
2    720575940899779727
3    720575940888561868
4    720575940928281596
Name: root_id, dtype: string
string
    tag          pt_position             root_id  soma_volume_um3  \
382   u  [23207, 2722, 1142]  720575940894614067              NaN   

     soma_volume_15um_um3  soma15_bbox_x_um  soma15_bbox_y_um  \
382           1202.545242            24.616            18.598   

     soma15_bbox_z_um  soma15_component_faces  soma15_component_vertices  ...  \
382            24.888                199903.0                   100210.0  ...   

     soma15_n_components       soma15_status soma15_watertight_before  \
382                430.0  success_watertight                    False   

    soma15_watertight_after soma15_raw_volume_um3  soma15_repair_volume_um3  \
382                    True           2085.211465               1202.545242   

     soma_area_xy_15um_um2  soma15_area_polygon_count  \
382             121.926715                      

In [30]:
import pandas as pd

# --------------------------------------------------
# Load corrected soma CSV
# --------------------------------------------------

soma_df = pd.read_csv(
    "superficial_in_soma_volume_15um_updated_corrected.csv",
    dtype={"root_id": "string"},
    low_memory=False,
)

print("=== soma_df ===")
print("root_id column dtype:", soma_df["root_id"].dtype)
print()

for idx, value in soma_df["root_id"].head(10).items():
    print(
        f"row {idx}: "
        f"value={value!r}, "
        f"Python type={type(value).__name__}"
    )



=== soma_df ===
root_id column dtype: string

row 0: value='720575940891783079', Python type=str
row 1: value='720575940889785957', Python type=str
row 2: value='720575940899779727', Python type=str
row 3: value='720575940888561868', Python type=str
row 4: value='720575940928281596', Python type=str
row 5: value='720575940917797515', Python type=str
row 6: value='720575940851357043', Python type=str
row 7: value='720575940930892621', Python type=str
row 8: value='720575940905042134', Python type=str
row 9: value='720575940918311989', Python type=str
